### 加载数据

In [2]:
import inspect
import numpy as np
import polars as pl
import pandas as pd
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from log_result import init_logger, log_result, log_print, read_log
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")
init_logger("WalkForward+ParameterSearch+Extremes")   

[log] 已绑定: D:\machine-learning-for-trading\case_studies\lazy_trading\logs\WalkForward+ParameterSearch+Extremes.log


WindowsPath('D:/machine-learning-for-trading/case_studies/lazy_trading/logs/WalkForward+ParameterSearch+Extremes.log')

In [3]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 市场基准
bench_symbol = "510300.SH"
bench = prices[bench_symbol]
prices = prices.drop(columns=[bench_symbol])  # 基准移出资产池
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# 中性化
X_net = X.sub(prices_to_returns(bench.to_frame("bench"), drop_inceptions_nan=False)["bench"], axis=0)
# Inf值检查
inf_cols = X_net.columns[np.isinf(X_net).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X_net = X_net.drop(columns=(bad := X_net.columns[(X_net.abs() > 0.25).any()])); 
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

中性化后相关性分布整体下移（中位数从高正相关降至 0.143）且正负两端均出现变化：强正相关配对减少约 40%，负相关配对从 0 增至千级。原参数网格（nondomin −0.3~−0.5 空转、correlate 0.1~0.5 高剔除）是按"市场因子主导的强正相关世界"标定的，直接复用确实不适配。建议：nondomin__threshold 收敛至 [-0.5, -0.4]、correlate__threshold 下调至 [0.15, 0.3]，同时计算规模恢复到原始口径水平。

## 相关性分布实测

| 配对相关性分布 | 原始 X | 超额 X_net | 变化 |
|---------------|--------|-----------|------|
| corr > 0.5 | 12,668 对 | 7,791 对 | **−38.5%** |
| corr > 0.3 | 27,712 对 | 16,726 对 | **−39.6%** |
| corr > 0.1 | 42,009 对 | 43,438 对 | +3.4% |
| corr < −0.3 | 0 对 | 1,269 对 | 0 → 1,269 |
| 中位数相关性 | 高（正相关主导） | 0.143 | 整体下移 |

In [133]:
import optuna
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure,ExtraRiskMeasure
from sklearn.pipeline import Pipeline
from skfolio.metrics import make_scorer
from skfolio.optimization import EqualWeighted
from Pre_selection import DropTailCorrelated
#from skfolio.pre_selection import SelectKExtremes
from Pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated
from skfolio.model_selection import WalkForward
from skfolio.model_selection import cross_val_predict, CombinatorialPurgedCV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
    - purged_size=0：训练结束与测试开始无缝衔接。
    - purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。
    - 建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。

- 训练集扩展与尾部数据处理
    - expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
    - reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。

#### WalkForward + Best Parameter

**test=252（年调仓周期）实验汇总**

| # | 时间 | train | 目标 | min_n | thr | corr | k | fitness | 年化收益 | ASR | MAD | MDD | 持仓 | F/P |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 3 | 01:34 | 756 | 0× | 25 | -0.5 | 0.3 | 0.3* | m-m-mdd-cvar | 16.39% | **1.23** | 0.61% | 25.21% | 6.5 |
| 22 | 21:24 | 756 | 0× | 25 | -0.5 | 0.3 | 0.3 | m-m-mdd-cvar | 16.39% | **1.23** | 0.61% | 25.21% | 6.5 |
| 25 | 22:37 | 252 | 1× | 25 | -0.3 | 0.1 | 0.5 | m-m-mdd-cvar-sr | 8.32% | 1.15 | 0.28% | 12.79% | 3.1 | P |
| 23 | 21:49 | 756 | 1× | 10 | -0.4 | 0.4 | 0.5 | m-semi-avgdd | 10.69% | 1.12 | 0.42% | 20.79% | 6.8 |
| 5 | 10:52 | 252 | 0× | 20 | -0.5 | 0.1 | 0.3* | m-v-avgdd | 9.04% | 1.02 | 0.33% | 10.40% | 2.4 | P |
| 20 | 21:07 | 252 | 0× | 10 | -0.3 | 0.1 | 0.1 | m-m-mdd-cvar-sr | 14.96% | 1.02 | 0.61% | 17.45% | 2.2 | P |
| 21 | 21:16 | 504 | 0× | 15 | -0.4 | 0.2 | 0.5 | m-v-avgdd | 7.86% | 0.96 | 0.33% | 11.59% | 4.1 |
| 24 | 22:27 | 504 | 1× | 10 | -0.3 | 0.1 | 0.5 | m-v-avgdd | 10.22% | 0.94 | 0.42% | 13.11% | 2.4 | P |
| 2 | 01:29 | 756 | 0× | 5 | -0.3 | 0.4 | 0.2* | mean-var | 11.98% | 0.91 | 0.58% | 21.99% | 4.4 |
| 4 | 10:42 | 504 | 0× | 10 | -0.4 | 0.1 | 0.3* | m-m-avgdd-cvar-sr | 9.63% | 0.76 | 0.47% | 14.99% | 2.4 | P |
| 1 | 01:22 | 1008 | 0× | 20 | -0.5 | 0.4 | 0.2* | m-m-mdd-cvar-sr | 9.73% | 0.71 | 0.63% | 34.44% | 7.6 |

fitness 简写：m-m-mdd-cvar = mean-mad-maxdd-cvar；m-m-mdd-cvar-sr = mean-mad-maxdd-cvar-sharpe；m-m-avgdd-cvar-sr = mean-mad-avgdd-cvar-sharpe；m-semi-avgdd = mean-semideviation-avgdd；m-v-avgdd = mean-variance-avgdd

In [80]:
from walkforward_parameter_search import FITNESS_MEASURES

In [86]:
def build_model(min_n=20, thr=-0.5, corr=0.1, k=0.3, fitness='mean-variance'):
    model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("extremes", SelectKExtremes(k=k, measure=ExtraRiskMeasure.KURTOSIS, highest=False)),
        ("nondomin", SelectNonDominated(min_n_assets=min_n, threshold=thr,
                        fitness_measures=FITNESS_MEASURES[fitness])),
        ("correlate", DropCorrelated(threshold=corr, absolute=False)),
        ("optimization", EqualWeighted()),
    ])
    return model

In [109]:
# ===== test=252（年调仓）11 个最优配置批量复现，生成 mpt 结果列表 =====
# 配置来源：WalkForward+ParameterSearch+Extremes.log 中 test_size=252 的 11 轮最优参数
# 字段: (轮次, train_size, min_n_assets, threshold, corr_threshold, k, fitness)

test252_configs = [
    #("#1 ", 1008, 20, -0.5, 0.4, 0.2, "mean-mad-maxdd-cvar-sharpe"),   # 阶段A k固定
    #("#2 ",  756,  5, -0.3, 0.4, 0.2, "mean-variance"),                # 阶段A k固定
    ("#4 ",  504, 10, -0.4, 0.1, 0.3, "mean-mad-avgdd-cvar-sharpe"),   # 阶段A k固定
    ("#5 ",  252, 20, -0.5, 0.1, 0.3, "mean-variance-avgdd"),          # 阶段A k固定
    ("#20",  252, 10, -0.3, 0.1, 0.1, "mean-mad-maxdd-cvar-sharpe"),
    #("#21",  504, 15, -0.4, 0.2, 0.5, "mean-variance-avgdd"),
    #("#22",  756, 25, -0.5, 0.3, 0.3, "mean-mad-maxdd-cvar"),
    #("#23",  756, 10, -0.4, 0.4, 0.5, "mean-semideviation-avgdd"),
    ("#24",  504, 10, -0.3, 0.1, 0.5, "mean-variance-avgdd"),
    ("#25",  252, 25, -0.3, 0.1, 0.5, "mean-mad-maxdd-cvar-sharpe"),
]

# build_model 内部引用全局 cv：循环内对 cv 重新绑定即可逐组生效
mpt_results = []            # [(轮次标签, MultiPeriodPortfolio), ...]
for label, train_size, min_n, thr, corr, k, fitness in test252_configs:
    model = build_model(min_n=min_n, thr=thr, corr=corr, k=k, fitness=fitness)
    cv = WalkForward(train_size=train_size, test_size=252,
                     purged_size=1, reduce_test=True)   # 与日志 WalkForward 配置一致
    mpt = cross_val_predict(model, X, cv=cv, n_jobs=4)
    mpt.name = label
    mpt_results.append(mpt)
    print(f"===== {label}  train={train_size}/252  {fitness} =====")

===== #4   train=504/252  mean-mad-avgdd-cvar-sharpe =====
===== #5   train=252/252  mean-variance-avgdd =====
===== #20  train=252/252  mean-mad-maxdd-cvar-sharpe =====
===== #24  train=504/252  mean-variance-avgdd =====
===== #25  train=252/252  mean-mad-maxdd-cvar-sharpe =====


In [ ]:
Population(mpt_results).plot_cumulative_returns()

### CPCV路径生成+LSH路径采样

In [142]:
from cpcv_analysis import discrete_lhs_safe
from wf_cpcv_search import build_test_parts

In [140]:
def fasted_walkforward_cpcv(
    model,
    X: pd.DataFrame,
    train_size: int,
    test_size: int,
    n_folds: int,
    n_test_folds: int,
    out_purged_size: int = 0,
    reduce_test: bool = False,
    expand_train: bool = False,
    in_purged_size: int = 2,
    in_embargo_size: int = 2,
    verbose: bool = True,
):
    """WalkForward(外) × CombinatorialPurgedCV(内) 嵌套交叉验证。

    外循环: WalkForward 滚动窗口, 每个 fold 把 train+test 拼成时间连续块;
    内层: 对整个窗口直接调用 cross_val_predict, 由它完成 CPCV 各 split 的
          预筛选→拟合→测试块预测, 不再单独对训练集做预测。

    返回:
        result: {outer_fold: {path_id: MultiPeriodPortfolio}}
        path_result: {outer_fold: {path_id: [Portfolio, ...]}}
    """
    outer_cv = WalkForward(
        test_size=test_size,
        train_size=train_size,
        purged_size=out_purged_size,
        reduce_test=reduce_test,
        expand_train=expand_train,
    )
    inner_cv = CombinatorialPurgedCV(
        n_folds=n_folds, n_test_folds=n_test_folds,
        purged_size=in_purged_size, embargo_size=in_embargo_size,
    )
    # （内部会对 estimator clone, 不会污染全局 selection_pipe）
    fold_mpts = {}
    fold_pops = {}

    for i, (train_index, test_index) in enumerate(outer_cv.split(X)):
        # ---- 1) train + test 拼成时间连续窗口 ----
        window_index = np.concatenate([train_index, test_index])
        X_window = X.iloc[window_index]
        # ---- 2) 内层 CPCV 一次完成: 返回 Population, 每个 path 一支 MultiPeriodPortfolio ----
        cvp = cross_val_predict(
            model, X_window, cv=inner_cv, n_jobs=4,
            portfolio_params=dict(name=f"WF{i}"),
        )
        # ---- 3) 按 path_id 组织结果 ----
        fold_mpts[i] = {pid: mptf for pid, mptf in enumerate(cvp)}         # MultiPeriodPortfolio
        fold_pops[i] = {pid: list(mptf)[-n_test_folds:] for pid, mptf in enumerate(cvp)}   # Portfolio

        if verbose:
            print(
                f"Outer fold {i}: window={len(X_window)} obs | "
                f"inner splits={inner_cv.get_n_splits(X_window)} | "
                f"paths={len(cvp)}"
            )

    return fold_mpts, fold_pops


In [170]:
label, train_size, min_n, thr, corr, k, fitness = ("#20",  252, 10, -0.3, 0.1, 0.1, "mean-mad-maxdd-cvar-sharpe")
model = build_model(min_n=min_n, thr=thr, corr=corr, k=k, fitness=fitness)

In [175]:
res, paths = fasted_walkforward_cpcv(model,
    X, train_size=train_size, test_size=252, n_folds=int(train_size+252)//(252//4), n_test_folds=4
)

Outer fold 0: window=504 obs | inner splits=70 | paths=35
Outer fold 1: window=504 obs | inner splits=70 | paths=35
Outer fold 2: window=504 obs | inner splits=70 | paths=35
Outer fold 3: window=504 obs | inner splits=70 | paths=35
Outer fold 4: window=504 obs | inner splits=70 | paths=35
Outer fold 5: window=504 obs | inner splits=70 | paths=35
Outer fold 6: window=504 obs | inner splits=70 | paths=35
Outer fold 7: window=504 obs | inner splits=70 | paths=35
Outer fold 8: window=504 obs | inner splits=70 | paths=35


In [ ]:
all_test_path = build_test_parts(paths, 4)
samples = discrete_lhs_safe(all_test_path, 3000)

In [ ]:
Population([MultiPeriodPortfolio(sample, compounded=False) for sample in samples]).plot_cumulative_returns()


In [127]:
mpt[0].assets

array(['159937.SZ', '159941.SZ', '161820.SZ'], dtype=object)

In [116]:
population = Population([portfolio for portfolio in mpt])

In [ ]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)